# Where the 0.94651 comes from — the anchor file changed, not the recipe

On September 16 the top public score moved from 0.94650 to 0.94651, credited to a Lasso meta-stack
blended at 5% into [@jazivxt's anchor](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline).
I spent a day of submissions finding out which part did it. It was the anchor.

jazivxt's dataset keeps the file name `submission_latest_best.csv` and replaces its contents whenever a
better submission lands. The README now reads *"Latest best public ROC AUC: 0.94651 — submission
56267408"*. Before September 16 the same file name held the 0.94649 submission.

| construction (same code, 5% Lasso, four rules, lexsort) | anchor behind `submission_latest_best.csv` | public LB |
|---|---|---|
| the published 0.94651 notebook's output file, resubmitted unchanged | 0.94651 (Sep 16) | **0.94651** |
| the same code re-run by me | 0.94649 (Sep 13) | 0.94650 |
| the same code, Lasso at 3% / 8% | 0.94649 (Sep 13) | 0.94650 / 0.94650 |

The cells below show the other half without needing any submission: run on today's anchor with the
Lasso file as it is published now, that code reproduces the 0.94651 output **byte for byte**. Swap in the
September 13 anchor and the same code drops a digit on the board.

In [ ]:
import glob, warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
warnings.filterwarnings("ignore")

ROOT = "/kaggle/input"
ID, TARGET = "id", "Will_Buy_EV"

def find_one(*pats):
    for p in pats:
        h = sorted(glob.glob(f"{ROOT}/**/{p}", recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(" | ".join(pats))

rk = lambda v: rankdata(v, method="average") / len(v)
test = pd.read_csv(find_one("playground-series-s6e9/test.csv", "test.csv")).sort_values(ID).reset_index(drop=True)
ids, N = test[ID].to_numpy(), len(test)

def load(path, col=None):
    d = pd.read_csv(path).sort_values(ID)
    assert len(d) == N
    return d[col or [c for c in d.columns if c != ID][-1]].to_numpy(dtype=float)

anchor = load(find_one("zoom-zoom-baseline/submission_latest_best.csv", "submission_latest_best.csv"))
lasso = pd.read_csv(find_one("lasso_test_predictions.csv")).sort_values(ID)["lasso_test"].to_numpy(dtype=float)
published = pd.read_csv(find_one("submission_blend.csv")).sort_values(ID)
assert np.array_equal(published[ID].to_numpy(), ids)
published = published[TARGET].to_numpy(dtype=float)
print("today's anchor: ties =", N - len(np.unique(anchor)))

## 1. Today's anchor reproduces the 0.94651 exactly

In [ ]:
def lexrank(primary, secondary):
    o = np.lexsort((secondary, primary))
    r = np.empty(N); r[o] = np.arange(1, N + 1)
    return (r - 0.5) / N

inc = test.Annual_Income_USD.to_numpy(float); km = test.Daily_Commute_km.to_numpy(float)
env1 = test.Environmental_Concern_Level.to_numpy() == 1
no_sub = test.Subsidy_Available.astype(str).to_numpy() == "No"
anx_mh = test.Range_Anxiety_Level.isin(["Medium", "High"]).to_numpy()
rules = {"income >= 170,537 (+10)": (inc >= 170537, 10.0), "31,004-41,970 (-10)": ((inc >= 31004) & (inc <= 41970), -10.0),
         "commute >= 83 (-5)": (km >= 83, -5.0), "30k, no subsidy, env1 or anxiety M/H (-5)": ((inc == 30000) & no_sub & (env1 | anx_mh), -5.0)}
shift = sum(np.where(m, s, 0.0) for m, s in rules.values())

rebuilt = lexrank(rk(anchor) + 0.05 * rk(lasso) + shift, lasso)       # the published recipe, verbatim
print(f"max |rebuilt - published| = {np.abs(rebuilt - published).max():.2e}")
print(f"spearman(rebuilt, published) = {spearmanr(rebuilt, published).statistic:.9f}")
print(f"spearman(published, anchor alone) = {spearmanr(published, anchor).statistic:.6f}")

## 2. Why the add-ons are now no-ops

Tie-breaking and the four boundary rules were real improvements on the older anchor. The new one already
contains both: it has no ties at all, and it already places every rule cell at the extreme it belongs to.

In [ ]:
r = rk(anchor)
rows = [[k, int(m.sum()), np.median(r[m]), r[m].min(), r[m].max()] for k, (m, s) in rules.items()]
pd.DataFrame(rows, columns=["rule", "test rows", "anchor rank pct median", "min", "max"]).round(4)

The dead-income zone, for instance, sits below the 6th percentile in today's anchor; in the September 13
anchor it reached the 67th. Forcing those rows down used to fix something. Now it moves rows that are
already at the bottom.

## 3. What adding more does now (public LB, one submission each)

| on today's 0.94651 anchor | public LB |
|---|---|
| anchor alone (jazivxt's README) | 0.94651 |
| + 5% Lasso, rules, lexsort (the published notebook) | 0.94651 |
| + my 0.94650 recipe: 6% OOF stack + 4% RealMLP, rules, lexsort | **0.94650** |
| + 25% further along the Sep 13 → Sep 16 change | 0.94651 |
| + 50% further along that change | **0.94650** |
| + one step along jazivxt's published `frozen_direction` | 0.94651 |

Everything I add from outside either does nothing or costs a digit. The Sep 13 → Sep 16 change is the one
direction the board is known to reward, and following it further overshoots by half a step. That is
what an anchor tuned against the public board looks like: the next digit is inside jazivxt's own
optimisation, not in a new member.

## 4. The file submitted here

Today's anchor plus one step along the direction jazivxt published in
`optimizer_replay_assets/frozen_direction.npz`, sized like jazivxt's own probes (mean absolute change
about 9e-5). It scores 0.94651. It is included so the badge on this page is the current top, and every
input it reads is public.

In [ ]:
fz = np.load(find_one("frozen_direction.npz"))
direction = fz["direction"].astype(float)
assert direction.shape == (N,)
scale = 9.3e-5 / np.abs(direction).mean()
final = rk(anchor + scale * direction)
print(f"spearman(final, anchor) = {spearmanr(final, anchor).statistic:.7f} | mean |rank change| {np.abs(final - rk(anchor)).mean():.2e}")
pd.DataFrame({ID: ids, TARGET: final}).to_csv("submission.csv", index=False)
print("wrote submission.csv")

---
Credit: [@jazivxt](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline) for the anchor, the scored
history and the published optimizer direction;
[@sometimessubodh](https://www.kaggle.com/code/sometimessubodh/0-94651-breaking-0-94650-lasso-meta-stack) for the
Lasso meta-stack notebook and its [engine](https://www.kaggle.com/code/sometimessubodh/s6e9-6-model-meta-stacking-engine),
whose published output made the exact reproduction possible;
[@taeyangg4](https://www.kaggle.com/code/taeyangg4/s6e9-094649-multi-paradigm-lexsort-master) for the boundary rules.
The submissions and measurements are mine. If this saved you a day of chasing the wrong ingredient,
an upvote helps.